In [1]:
import sys
sys.path.append("/home/hweiner/Internship/CoMET_rams_devel/")

import CoCoMET
from microbench import MicroBench
import pandas as pd


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119

===============Welcome To CoCoMET===============

A toolkit of the Advanced Study of Cloud and Environment iNTerations (ASCENT) program.

This project was supported by the U.S. Department of Energy (DOE) Early Career Research Program, Atmospheric System Research (ASR) program, and the Office of Workforce Development for Teachers and Scientists (WDTS) under the Science Undergraduate Laboratory Internships Program (SULI).

If you are using this software for a publication, please cite: ####




In [2]:
# Make the config

CONFIG_str_parallel = """
# SETUP VARIABLES: These determine basic CoMET functionality

verbose: False # Whether to use verbose output
parallel_processing: True # [bool] Whether or not to use parallel processing for certain tasks
max_cores: 32 # Number of cores to use if parallel_processing==True; Enter None for unlimited

# Structered in this form:
# Observation Type:
#   path_to_data
#   additional_observation_parameters
#
#   tracker:
#       tracker_params
#
#       analysis:
#           analysis_variables

# WRF
wrf:
    path_to_data: "/D3/data/hweiner/CoMET_out/time_analysis/data/.cocomet_testing_datasets/WRF/wrfout*"
    
    is_idealized: False
    #min_frame_index: 10 # 0-based indexing, inclusive
    #max_frame_index: 70 # 0-based indexing, inclusive

    feature_tracking_var: "TB" #DBZ, TB, WA, PR, Or other WRF Standard variable name (Case-Sensitive)
    segmentation_var: "TB"

    tobac:
        feature_id:
            threshold: [235, 219]
            target: "minimum"
            position_threshold: "weighted_diff"
            sigma_threshold: 0.5
            n_min_threshold: 200
            #detect_subset: {1:(29, 47)}

        linking:
            method_linking: "predict"
            adaptive_stop: 0.2
            adaptive_step: 0.95
            order: 1
            subnetwork_size: 10
            memory: 0
            v_max: 20
            time_cell_min: 120

        segmentation_2d:
            method: "watershed"
            target: 'maximum'
            threshold: 219
    
    moaap:
        tracking_save_path: "/D3/data/hweiner/CoMET_out/time_analysis/data/.cocomet_testing_datasets/WRF/moaap_tracking/test/"
    
    tams: 
        ctt_threshold: 235
        ctt_core_threshold: 219
        u_projection: 0
        parallel: False

        analysis_type: "cloud"
    
# NEXRAD
nexrad:
    path_to_data: "/D3/data/hweiner/CoMET_out/time_analysis/data/.cocomet_testing_datasets/NEXRAD/KVNX*"

    bounds: [-100, -95.5, 35.5, 38.5] # In form [lon_min,lon_max,lat_min,lat_max], OPTIONAL

    gridding: # OPTIONAL only if gridding is needed 
        gridding_save_path: "/D3/data/hweiner/CoMET_out/time_analysis/data/.cocomet_testing_datasets/NEXRAD/grids/"
        grid_shape: (40, 401, 401)
        grid_limits: ((500, 20000), (-200000., 200000.), (-200000., 200000.))
        
    feature_tracking_var: "DBZ" #DBZ, TB, WA, PR, Or other WRF Standard variable name (Case-Sensitive)
    segmentation_var: "DBZ"

    tobac:
        feature_id:
            threshold: [30,40,50,60]
            target: "minimum"
            position_threshold: "weighted_diff"
            sigma_threshold: 0.5
            n_min_threshold: 200
            #detect_subset: {1:(29, 47)}

        linking:
            method_linking: "predict"
            adaptive_stop: 0.2
            adaptive_step: 0.95
            order: 1
            subnetwork_size: 10
            memory: 0
            v_max: 20
            time_cell_min: 120

        segmentation_2d:
            method: "watershed"
            height: 2
            threshold: 30
    
    moaap:
        tracking_save_path: "/D3/data/hweiner/CoMET_out/time_analysis/data/.cocomet_testing_datasets/WRF/moaap_tracking/test/"
    
    tams: 
        ctt_threshold: 235
        ctt_core_threshold: 219
        u_projection: 0
        parallel: False

        analysis_type: "cloud"

# GOES
goes:
    path_to_data: "/D3/data/hweiner/CoMET_out/time_analysis/data/.cocomet_testing_datasets/GOES/*"

    bounds: [-100, -95.5, 35.5, 38.5] # In form [lon_min,lon_max,lat_min,lat_max]
    
    feature_tracking_var: "TB"
    segmentation_var: "TB"
    
    tobac:
        feature_id:
            threshold: [235, 219]
            target: "minimum"
            position_threshold: "weighted_diff"
            sigma_threshold: 0.5
            n_min_threshold: 200
            #detect_subset: {1:(29, 47)}

        linking:
            method_linking: "predict"
            adaptive_stop: 0.2
            adaptive_step: 0.95
            order: 1
            subnetwork_size: 10
            memory: 0
            v_max: 20
            time_cell_min: 120

        segmentation_2d:
            method: "watershed"
            target: 'maximum'
            threshold: 219
    
    moaap:
        tracking_save_path: "/D3/data/hweiner/CoMET_out/time_analysis/data/.cocomet_testing_datasets/WRF/moaap_tracking/test/"
    
    tams: 
        ctt_threshold: 235
        ctt_core_threshold: 219
        u_projection: 0
        parallel: False

        analysis_type: "cloud"

standard_radar:
    path_to_data: "/D3/data/hweiner/CoMET_out/time_analysis/data/.cocomet_testing_datasets/GOAMAZON_RADAR/*"
    
    min_frame_index: 0 # 0-based indexing, inclusive
    max_frame_index: 44 # 0-based indexing, inclusive

    feature_tracking_var: "dbz"
    segmentation_var: "dbz"
    
    tobac:
        feature_id:
            threshold: [30,40,50,60]
            target: "maximum"
            position_threshold: "weighted_diff"
            sigma_threshold: 0.5
            n_min_threshold: 20
        
        linking: 
            method_linking: "predict"
            adaptive_stop: 0.2
            adaptive_step: 0.95
            order: 1
            subnetwork_size: 10
            memory: 1
            v_max: 20
        
        segmentation_2d:
            height: 2 #km
            method: "watershed"
            threshold: 15    
"""

In [3]:
# Create a copy of the previous config but without parallel processing
from copy import deepcopy
CONFIG_parallel = CoCoMET.CoCoMET_load(CONFIG_string = CONFIG_str_parallel)
print(CONFIG_parallel)
CONFIG_non_parallel = deepcopy(CONFIG_parallel)
CONFIG_non_parallel["parallel_processing"] = False
CONFIG_non_parallel["max_cores"] = 1

print(CONFIG_non_parallel)

{'verbose': False, 'parallel_processing': True, 'max_cores': 32, 'wrf': {'path_to_data': '/D3/data/hweiner/CoMET_out/time_analysis/data/.cocomet_testing_datasets/WRF/wrfout*', 'is_idealized': False, 'feature_tracking_var': 'TB', 'segmentation_var': 'TB', 'tobac': {'feature_id': {'threshold': [235, 219], 'target': 'minimum', 'position_threshold': 'weighted_diff', 'sigma_threshold': 0.5, 'n_min_threshold': 200}, 'linking': {'method_linking': 'predict', 'adaptive_stop': 0.2, 'adaptive_step': 0.95, 'order': 1, 'subnetwork_size': 10, 'memory': 0, 'v_max': 20, 'time_cell_min': 120}, 'segmentation_2d': {'method': 'watershed', 'target': 'maximum', 'threshold': 219}}, 'moaap': {'tracking_save_path': '/D3/data/hweiner/CoMET_out/time_analysis/data/.cocomet_testing_datasets/WRF/moaap_tracking/test/'}, 'tams': {'ctt_threshold': 235, 'ctt_core_threshold': 219, 'u_projection': 0, 'parallel': False, 'analysis_type': 'cloud'}}, 'nexrad': {'path_to_data': '/D3/data/hweiner/CoMET_out/time_analysis/data/.

In [4]:
# Create the microbench
test_bench_parallel = MicroBench(outfile="/D3/data/hweiner/CoMET_out/time_analysis/output/parallel.out")
test_bench_non_parallel = MicroBench(outfile="/D3/data/hweiner/CoMET_out/time_analysis/output/non_parallel.out")

In [5]:
@test_bench_parallel
def run_cocomet_parallel():
    output = CoCoMET.CoCoMET_start(CONFIG = CONFIG_parallel)
    return output

In [6]:
@test_bench_non_parallel
def run_cocomet_non_parallel():
    output = CoCoMET.CoCoMET_start(CONFIG = CONFIG_non_parallel)
    return output

In [ ]:
out_parallel = run_cocomet_parallel()
out_non_parallel = run_cocomet_non_parallel()

In [8]:
results_parallel = pd.read_json(test_bench_parallel.outfile, lines=True)
results_non_parallel = pd.read_json(test_bench_non_parallel.outfile, lines=True)

In [9]:
results_parallel

# In a parallelised environment, CoCoMET takes ~54s to run

,timestamp_tz,duration_counter,function_name,run_durations,start_time,finish_time
0,UTC,perf_counter,run_cocomet_parallel,[53.933849869761616],2025-07-09 02:30:50.245038+00:00,2025-07-09 02:31:44.178913+00:00
1,UTC,perf_counter,run_cocomet_parallel,[54.60690844198689],2025-07-09 03:12:27.588799+00:00,2025-07-09 03:13:22.195724+00:00


In [10]:
results_non_parallel

# As opposed to a non-parallelised environment which takes ~486s 
# to run the same data with the same trackers

,timestamp_tz,duration_counter,function_name,run_durations,start_time,finish_time
0,UTC,perf_counter,run_cocomet_non_parallel,[487.7650196570903],2025-07-09 02:31:44.179431+00:00,2025-07-09 02:39:51.944460+00:00
1,UTC,perf_counter,run_cocomet_non_parallel,[485.085939175915],2025-07-09 03:13:22.196064+00:00,2025-07-09 03:21:27.282013+00:00
